# Модуль 4. Кеширование и CDN

## Теоретический разбор и анализ результатов практической работы

**Курс:** Разработка высоконагруженных сервисов  
**Модуль:** 4. Кеширование и CDN  
**Тема:** Уровни кеширования, стратегии, инвалидация, Redis, cache stampede

---

## Содержание блокнота

1. Теоретическое введение: зачем нужно кеширование
2. Уровни кеширования
3. Стратегии кеширования: cache-aside, write-through, write-behind
4. Инвалидация кеша и Cache Stampede (dogpile effect)
5. Redis: архитектура и eviction policies
6. Redis в Python: redis.asyncio и типичные паттерны
7. CDN: как работает и зачем нужен
8. Практика: добавление Redis в проект
9. Практика: кеширование каталога товаров
10. Практика: измерение Cache Hit Ratio
11. Практика: нагрузочное тестирование с холодным и тёплым кешем
12. Метрики Locust после кеширования
13. Выводы

**Предварительное условие:** система из модуля 3 запущена (`docker compose up -d`), PostgreSQL наполнен данными, Redis добавлен в docker-compose.

## 1. Теоретическое введение: зачем нужно кеширование

### 1.1. Проблема, которую решает кеш

В модуле 3 мы увидели, что добавление БД снижает RPS и увеличивает latency. Причина — каждая страница каталога требует запроса к PostgreSQL, а это:

- Disk I/O (~100 мкс на SSD)
- Парсинг SQL (~50 мкс)
- Планирование запроса (~100 мкс)
- Выполнение и сериализация результатов (~500 мкс – 10 мс)

**Итог:** 1–50 мс на запрос, даже если данные не менялись с прошлого раза.

### 1.2. Аналогия

Представьте библиотеку. Каждый раз, когда посетитель просит книгу, библиотекарь идёт в подвал, ищет её среди 100 000 других, поднимается наверх — 5 минут. Если та же книга запрашивается 100 раз в день, библиотекарь тратит 8 часов только на поиск.

**Кеш — это полка с самыми популярными книгами прямо у стойки.** Первый раз библиотекарь всё ещё идёт в подвал, но следующие 99 раз книга уже под рукой. Время ответа: 5 секунд.

### 1.3. Экономика кеша

| Параметр | Без кеша | С кешем (90% hit rate) |
|---|---|---|
| Средняя latency | 10 мс | 1.9 мс |
| Нагрузка на БД | 100% | 10% |
| RPS | 1000 | 8000+ |
| Стоимость инфраструктуры | Высокая | Ниже в 3–5 раз |

**Правило: 90% трафика — это чтение одних и тех же данных.** Именно для этих 90% и нужен кеш.

### 1.4. Закон Литтла и кеш

Если время ответа уменьшается в 5 раз (с 10 мс до 2 мс), то пропускная способность при том же количестве воркеров растёт в 5 раз. Это прямое следствие закона Литтла:

```
L = λ × W
L — число запросов в системе
λ — интенсивность поступления (RPS)
W — время обработки (latency)

При фиксированном L (ресурсы сервера):
λ = L / W

Уменьшили W в 5 раз → λ вырос в 5 раз
```

## 2. Уровни кеширования

### 2.1. Полная карта уровней

```
Клиент
  └── HTTP Cache (браузер): Cache-Control, ETag, Last-Modified
        │
CDN (CloudFlare, CloudFront)
  └── Кеш статики и API-ответов на edge-серверах
        │
Nginx / Reverse Proxy
  └── proxy_cache: кеш ответов backend
        │
Application Layer (Redis, Memcached)
  └── Кеш результатов запросов и вычислений
        │
Database Layer
  └── shared_buffers, OS page cache, materialized views
```

### 2.2. Сравнение уровней

| Уровень | Скорость | Стоимость | Инвалидация | Где использовать |
|---|---|---|---|---|
| **HTTP Cache (браузер)** | <1 мс | Бесплатно | По URL + версии | Статика, неизменяемые данные |
| **CDN** | 5–30 мс | Средняя | По TTL и purge API | Изображения, JS, CSS, API-ответы |
| **Nginx proxy_cache** | <1 мс | Низкая | По ключу и времени | API-ответы, разделяемые всеми |
| **Redis / Memcached** | 0.1–1 мс | Средняя | Точная по ключу | Персональные данные, каталог |
| **Database cache** | 10 мкс – 10 мс | Бесплатно | Автоматически | Горячие данные |

### 2.3. Кеш-иерархия: правило холодного/тёплого

**Холодный кеш (cold cache):** данных нет, запрос идёт в источник. Медленно.

**Тёплый кеш (warm cache):** данные есть, отдаются мгновенно. Быстро.

**Прогрев (warmup):** после деплоя или рестарта кеш пустой. Первые запросы медленные. Решения:
- **Pre-warming** — скрипт заранее заполняет кеш популярными ключами
- **Read-through** — первый пользователь платит ценой медленного запроса, остальные получают из кеша
- **Старый кеш + обновление в фоне** — отдаём устаревшие данные, пока обновляем

## 3. Стратегии кеширования

### 3.1. Cache-Aside (Lazy Loading)

**Самый популярный паттерн.** Приложение само управляет кешем.

| Плюсы | Минусы |
|---|---|
| Простота | Кеш не обновляется при изменении данных в БД |
| Только нужные данные | Первый запрос всегда медленный |
| Устойчив к падению Redis (работаем без кеша) | Возможна рассинхронизация |

Пример:

In [2]:
# Cache-Aside (Lazy Loading) — иллюстративный пример
async def get_product_cache_aside(product_id: int, redis, db):
    """Читаем товар: сначала кеш, при промахе — БД, затем пишем в кеш."""
    # 1. Пробуем получить из кеша
    cached = await redis.get(f"product:{product_id}")
    if cached:
        return json.loads(cached)  # HIT
    
    # 2. Промах — идём в БД
    product = await db.get(Product, product_id)  # MISS
    if product is None:
        return None
    
    # 3. Записываем в кеш на 300 секунд
    await redis.setex(f"product:{product_id}", 300, json.dumps(product, default=str))
    return product

### 3.2. Write-Through

**Запись идёт и в БД, и в кеш одновременно.**

| Плюсы | Минусы |
|---|---|
| Кеш всегда актуален | Двойная запись (медленнее) |
| Нет рассинхронизации | Записываем данные, которые могут не читаться |

In [ ]:
# Write-Through — иллюстративный пример
async def update_product_write_through(product_id: int, data: dict, redis, db):
    """Обновление товара: пишем в БД и в кеш одновременно."""
    async with db.begin():
        await db.update(Product, product_id, data)
        await redis.setex(f"product:{product_id}", 300, json.dumps(data, default=str))

### 3.3. Write-Behind (Write-Back)

**Запись сначала в кеш, потом асинхронно в БД.**

| Плюсы | Минусы |
|---|---|
| Очень быстро для записи | Риск потери данных при падении |
| Батчинг записей | Сложность реализации |

In [ ]:
# Write-Behind — иллюстративный пример
async def update_product_write_behind(product_id: int, data: dict, redis, message_queue):
    """Обновление товара: пишем в кеш сразу, в БД — через очередь."""
    await redis.set(f"product:{product_id}", json.dumps(data, default=str))
    await message_queue.publish("product.updated", {"id": product_id, "data": data})

### 3.4. Сравнение стратегий

| Стратегия | Консистентность | Скорость записи | Сложность | Когда использовать |
|---|---|---|---|---|
| **Cache-Aside** | Низкая | Быстрая | Низкая | Чтение >> запись |
| **Write-Through** | Высокая | Медленная | Средняя | Баланс чтения/записи |
| **Write-Behind** | Средняя | Очень быстрая | Высокая | Запись >> чтение |
| **Refresh-Ahead** | Средняя | — | Средняя | Горячие ключи |

**В нашем проекте будем использовать Cache-Aside** — он проще и покрывает 90% сценариев.

## 4. Инвалидация кеша и Cache Stampede

### 4.1. Почему инвалидация — самая сложная проблема

«В информатике есть только две сложные проблемы: инвалидация кеша и придумывание имён.» — Фил Карлтон

### 4.2. Стратегии инвалидации

| Стратегия | Как работает | Когда использовать |
|---|---|---|
| **TTL (Time To Live)** | Ключ удаляется через N секунд | Данные, где устаревание допустимо |
| **Explicit delete** | Приложение удаляет ключ при обновлении | Точная инвалидация |
| **Version keys** | Ключ содержит версию (`user:42:v5`) | Часто обновляемые сущности |
| **Event-driven** | Публикуем событие при изменении, все воркеры инвалидируют | Микросервисы |
| **Write-through** | Кеш обновляется вместе с БД | Высокая консистентность |

### 4.3. Cache Stampede (Dogpile Effect)

**Проблема:** популярный ключ истекает. 1000 запросов одновременно обнаруживают промах и все идут в БД. БД не выдерживает.

```
TTL истёк → 1000 запросов → 1000 MISS → 1000 запросов в БД → перегрузка
```

### 4.4. Решения Cache Stampede

#### 4.4.1. Mutex / Lock (Single Flight)

Только один запрос идёт в БД, остальные ждут результат.

In [ ]:
# Mutex / Single Flight — иллюстративный пример
async def get_with_lock(key: str, fetch_fn, redis):
    """Защита от stampede: только один процесс идёт в источник."""
    cached = await redis.get(key)
    if cached:
        return json.loads(cached)
    
    # Пытаемся взять блокировку
    lock_key = f"lock:{key}"
    acquired = await redis.set(lock_key, "1", nx=True, ex=10)
    
    if not acquired:
        # Кто-то другой уже обновляет — ждём
        await asyncio.sleep(0.05)
        return await get_with_lock(key, fetch_fn, redis)
    
    try:
        data = await fetch_fn()
        await redis.setex(key, 60, json.dumps(data, default=str))
        return data
    finally:
        await redis.delete(lock_key)

#### 4.4.2. Probabilistic Early Expiration (XFetch)

Каждый запрос с некоторой вероятностью обновляет ключ **до** истечения TTL. Чем ближе к истечению, тем выше вероятность.

In [ ]:
# XFetch (Probabilistic Early Expiration) — иллюстративный пример
import random
import math

async def get_with_early_expiration(key: str, ttl: int, fetch_fn, redis, beta: float = 1.0):
    """Вероятностное обновление кеша до истечения TTL."""
    cached = await redis.get(key)
    if cached:
        payload = json.loads(cached)
        data = payload["data"]
        remaining_ttl = await redis.ttl(key)
        
        # Вероятность обновления растёт по мере приближения к TTL
        threshold = beta * math.log(random.random()) * -1
        if remaining_ttl > threshold:
            return data
    
    # Обновляем
    data = await fetch_fn()
    await redis.setex(key, ttl, json.dumps({"data": data}, default=str))
    return data

#### 4.4.3. Stale-While-Revalidate

Отдаём устаревшие данные, а в фоне обновляем.

In [ ]:
# Stale-While-Revalidate — иллюстративный пример
import time

async def refresh_cache(key: str, fetch_fn, redis, ttl: int = 60):
    """Обновляет кеш из источника и возвращает свежие данные."""
    data = await fetch_fn()
    payload = {"data": data, "stored_at": time.time()}
    await redis.setex(key, ttl * 2, json.dumps(payload, default=str))
    return data


async def get_stale_while_revalidate(key: str, soft_ttl: int, fetch_fn, redis):
    """Отдаём устаревшие данные, обновляем в фоне."""
    cached = await redis.get(key)
    if cached:
        payload = json.loads(cached)
        age = time.time() - payload["stored_at"]
        
        if age < soft_ttl:
            return payload["data"]  # Свежие данные
        
        # Устарели, но можно отдать. Запускаем обновление в фоне
        asyncio.create_task(refresh_cache(key, fetch_fn, redis))
        return payload["data"]
    
    # Совсем нет кеша — синхронно обновляем
    return await refresh_cache(key, fetch_fn, redis)

### 4.5. Thundering Herd при старте

Похожая проблема: после рестарта сервиса кеш пустой, все запросы идут в БД. Решение: **pre-warming** при старте — скрипт прогревает популярные ключи, пока сервис ещё не принимает трафик.

## 5. Redis: архитектура и eviction policies

### 5.1. Почему Redis, а не Memcached

| Характеристика | Redis | Memcached |
|---|---|---|
| **Структуры данных** | String, Hash, List, Set, Sorted Set, Stream, HyperLogLog | Только String |
| **Персистентность** | RDB, AOF | Нет |
| **Репликация** | Да | Нет |
| **Кластеризация** | Redis Cluster | Клиентский шардинг |
| **Pub/Sub** | Да | Нет |
| **Скрипты** | Lua | Нет |
| **TTL на ключ** | Да | Да |
| **Потребление памяти** | Больше | Меньше |

**Redis выбирают, когда нужны не только кеш, но и структуры данных, pub/sub, очереди, счётчики.**

### 5.2. Eviction policies

| Политика | Что удаляет | Когда использовать |
|---|---|---|
| **noeviction** | Ничего (ошибка при записи) | Redis не для кеша |
| **allkeys-lru** | Наименее недавно использованные | Универсальный кеш |
| **allkeys-lfu** | Наименее часто использованные | Есть «горячие» и «холодные» ключи |
| **allkeys-random** | Случайные | Равномерный доступ |
| **volatile-lru** | LRU среди ключей с TTL | Часть данных — персистентная |
| **volatile-lfu** | LFU среди ключей с TTL | То же, но с частотой |
| **volatile-random** | Случайные среди с TTL | Редко |
| **volatile-ttl** | С наименьшим оставшимся TTL | Когда важно удалять раньше истекающие |

**Для кеша каталога товаров выбираем `allkeys-lru`.**

### 5.3. Конфигурация Redis для кеша

```conf
# redis.conf
maxmemory 512mb
maxmemory-policy allkeys-lru
save ""                    # Отключить RDB-снимки
appendonly no              # Отключить AOF
tcp-keepalive 300
timeout 0
```

**Важно:** если Redis используется только как кеш, персистентность не нужна.

### 5.4. Мониторинг Redis

**Cache Hit Ratio:**

```
hit_ratio = keyspace_hits / (keyspace_hits + keyspace_misses)
```

**Норма:** > 80% для универсального кеша, > 95% для горячего каталога.

## 6. Redis в Python: redis.asyncio и типичные паттерны

### 6.1. Установка

```bash
pip install redis
```

### 6.2. Асинхронное подключение

Ниже — рабочий пример подключения к Redis-контейнеру из блокнота.

In [ ]:
import redis.asyncio as aioredis
import asyncio
import json

# Подключение к Redis (порт 6379 проброшен на localhost)
redis_client = aioredis.Redis(
    host="localhost",
    port=6379,
    db=0,
    decode_responses=True,       # Возвращать str, а не bytes
    socket_timeout=2,
    socket_connect_timeout=2,
    max_connections=50,
)

# Проверка подключения
async def ping_redis():
    try:
        pong = await redis_client.ping()
        print(f"✓ Redis отвечает: {pong}")
    except Exception as e:
        print(f"✗ Redis недоступен: {e}")

await ping_redis()

### 6.3. Основные операции

Демонстрация всех базовых операций: строки, TTL, хеши, счётчики, pipeline, NX-блокировки.

In [ ]:
async def demo_basic_operations():
    """Демонстрация базовых операций с Redis."""
    
    # --- Строки ---
    await redis_client.set("demo:key", "value")
    value = await redis_client.get("demo:key")
    print(f"set/get: {value}")
    
    # С TTL 60 секунд
    await redis_client.setex("demo:ttl", 60, "expires in 60s")
    ttl = await redis_client.ttl("demo:ttl")
    print(f"setex: TTL = {ttl} сек")
    
    # --- Хеши ---
    await redis_client.hset("demo:user:42", mapping={"name": "Alice", "age": 30})
    user = await redis_client.hgetall("demo:user:42")
    print(f"hset/hgetall: {user}")
    
    # --- Инкремент (атомарный счётчик) ---
    await redis_client.delete("demo:counter")
    for _ in range(3):
        await redis_client.incr("demo:counter")
    counter = await redis_client.get("demo:counter")
    print(f"incr x3: {counter}")
    
    # --- Pipeline (батчинг) ---
    async with redis_client.pipeline() as pipe:
        pipe.set("demo:k1", "v1")
        pipe.set("demo:k2", "v2")
        pipe.set("demo:k3", "v3")
        await pipe.execute()
    keys = await redis_client.keys("demo:k*")
    print(f"pipeline: создано {len(keys)} ключей")
    
    # --- NX (установить, только если ключа нет) ---
    await redis_client.delete("demo:lock")
    first = await redis_client.set("demo:lock", "1", nx=True, ex=10)
    second = await redis_client.set("demo:lock", "1", nx=True, ex=10)
    print(f"NX: первая попытка = {first}, вторая = {second}")
    
    # --- Очистка демонстрационных ключей ---
    demo_keys = await redis_client.keys("demo:*")
    if demo_keys:
        await redis_client.delete(*demo_keys)
    print(f"Очищено {len(demo_keys)} демо-ключей")

await demo_basic_operations()

### 6.4. Ключи: naming convention

**Хорошая практика:** использовать префиксы через двоеточие.

```
product:42                # Товар #42
products:page:1           # Страница каталога
user:42:orders            # Заказы пользователя #42
session:abc123            # Сессия
lock:product:42           # Блокировка
ratelimit:ip:1.2.3.4      # Rate limiting
```

**Почему важно:** можно инвалидировать группу по паттерну, легко искать в RedisInsight, понятно по логам.

### 6.5. Сериализация

Redis хранит строки, но нам нужны объекты. Варианты:

| Формат | Скорость | Размер | Читаемость |
|---|---|---|
| **JSON** | Средняя | Средний | Отличная |
| **MessagePack** | Быстро | Меньше | Плохая |
| **Pickle** | Быстро | Средний | Небезопасно |
| **Protobuf** | Очень быстро | Минимальный | Плохая |

**Для проекта:** JSON. Просто, читаемо, достаточно быстро.

## 7. CDN: как работает и зачем нужен

### 7.1. Что такое CDN

**CDN (Content Delivery Network)** — это географически распределённая сеть серверов, которая кеширует контент ближе к пользователям.

```
Без CDN:
  Клиент в Новосибирске → Сервер в Москве (2000 км, 60 мс)

С CDN:
  Клиент в Новосибирске → Edge-сервер в Новосибирске (5 км, 5 мс)
                              ↓ (только при промахе)
                         Origin-сервер в Москве
```

### 7.2. Что кешируется в CDN

| Тип контента | TTL | Пример |
|---|---|---|
| **Статика с версией** | 1 год | `app.v123.js`, `logo.abc.png` |
| **Статика без версии** | 1 час – 1 день | `logo.png` |
| **API-ответы** | 10 сек – 5 мин | `/api/products` |
| **HTML-страницы** | 1–5 мин | Главная, категория |
| **Персональные данные** | НЕ кешируются | `/api/me` |

### 7.3. HTTP-заголовки кеширования

```http
Cache-Control: public, max-age=3600, s-maxage=7200, stale-while-revalidate=60
ETag: "abc123"
Last-Modified: Wed, 15 Jan 2025 12:00:00 GMT
Vary: Accept-Encoding, Accept-Language
```

**Расшифровка директив:**

| Директива | Значение |
|---|---|
| `public` | Можно кешировать везде (включая CDN) |
| `private` | Только в браузере (не в CDN) |
| `no-cache` | Кешировать, но валидировать перед использованием |
| `no-store` | Не кешировать вообще |
| `max-age=N` | Секунд до устаревания (для браузера) |
| `s-maxage=N` | Секунд до устаревания (для CDN, переопределяет max-age) |
| `stale-while-revalidate=N` | Отдаём устаревшее, пока обновляем в фоне |

### 7.4. Популярные CDN

| CDN | Особенности |
|---|---|
| **Cloudflare** | Самая большая сеть, бесплатный тариф, DDoS-защита |
| **AWS CloudFront** | Глубокая интеграция с AWS |
| **Fastly** | Программируемый edge (VCL, Compute@Edge) |
| **Akamai** | Крупнейшая сеть, enterprise |
| **Bunny CDN** | Дешёвый, простой |

**Для нашего проекта:** в учебных целях CDN не разворачиваем (локально), но проектируем API так, чтобы ответы можно было кешировать в CDN в будущем.

In [ ]:
# Установка зависимостей
!pip install redis requests pandas matplotlib --quiet

In [ ]:
import subprocess
import requests
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import time
import json
import re
from collections import Counter
from io import StringIO

NGINX_URL = "http://localhost:8080"
REDIS_CONTAINER = "redis"

print("Настройки загружены.")

## 8. Практика: добавление Redis в проект

### 8.1. Что нужно сделать

1. Добавить Redis в `docker-compose.yml`
2. Установить `redis` Python-пакет
3. Создать файл `app/cache.py` с redis_client
4. Реализовать функцию `cached()` — обёртку над Cache-Aside
5. Обернуть эндпоинты `/products` и `/products/{id}`
6. Реализовать инвалидацию при обновлении товара

### 8.2. Конфигурация Redis в docker-compose.yml

```yaml
services:
  redis:
    image: redis:7-alpine
    container_name: redis
    command: >
      redis-server
      --maxmemory 512mb
      --maxmemory-policy allkeys-lru
      --save ""
      --appendonly no
    ports:
      - "6379:6379"
    networks:
      - highload-net
    healthcheck:
      test: ["CMD", "redis-cli", "ping"]
      interval: 5s
      timeout: 3s
      retries: 5
```

### 8.3. Обёртка Cache-Aside (файл `app/cache.py`)

Код ниже — это содержимое файла `app/cache.py` в вашем FastAPI-проекте. Он не выполняется в блокноте, но используется приложением.

In [ ]:
# Содержимое файла app/cache.py (для проекта, не для блокнота)
cache_py_content = '''
import json
import redis.asyncio as redis
from typing import Any, Callable, Optional
import asyncio

redis_client = redis.Redis(
    host="redis", port=6379, db=0,
    decode_responses=True,
    socket_timeout=2,
)


async def get_json(key: str) -> Optional[Any]:
    try:
        raw = await redis_client.get(key)
        return json.loads(raw) if raw else None
    except Exception:
        return None


async def set_json(key: str, value: Any, ttl: int = 60) -> None:
    try:
        await redis_client.setex(key, ttl, json.dumps(value, default=str))
    except Exception:
        pass


async def invalidate(key: str) -> None:
    try:
        await redis_client.delete(key)
    except Exception:
        pass


async def cached(key: str, ttl: int, fetch_fn: Callable, lock_timeout: int = 10):
    cached_value = await get_json(key)
    if cached_value is not None:
        return cached_value
    
    lock_key = f"lock:{key}"
    try:
        acquired = await redis_client.set(lock_key, "1", nx=True, ex=lock_timeout)
    except Exception:
        acquired = True
    
    if not acquired:
        for _ in range(20):
            await asyncio.sleep(0.05)
            cached_value = await get_json(key)
            if cached_value is not None:
                return cached_value
    
    try:
        value = await fetch_fn()
        await set_json(key, value, ttl)
        return value
    finally:
        try:
            await redis_client.delete(lock_key)
        except Exception:
            pass
'''

# Сохраняем в файл (для копирования в проект)
with open("cache.py.example", "w", encoding="utf-8") as f:
    f.write(cache_py_content)

print("✓ Содержимое app/cache.py сохранено в cache.py.example")
print(f"Размер файла: {len(cache_py_content)} байт")

### 8.4. Использование в эндпоинтах FastAPI

Пример эндпоинтов с кешированием (содержимое `app/main.py` вашего проекта).

In [ ]:
# Пример эндпоинтов с кешированием (для проекта, не для блокнота)
endpoints_example = '''
@app.get("/products")
async def list_products(limit: int = 100, db: AsyncSession = Depends(get_db)):
    cache_key = f"products:limit:{limit}"
    
    async def fetch():
        result = await db.execute(select(Product).limit(limit))
        products = result.scalars().all()
        return [
            {"id": p.id, "sku": p.sku, "name": p.name,
             "price": str(p.price), "stock": p.stock, "category": p.category}
            for p in products
        ]
    
    return await cached(cache_key, ttl=60, fetch_fn=fetch)


@app.get("/products/{product_id}")
async def get_product(product_id: int, db: AsyncSession = Depends(get_db)):
    cache_key = f"product:{product_id}"
    
    async def fetch():
        product = await db.get(Product, product_id)
        if not product:
            return None
        return {
            "id": product.id, "sku": product.sku, "name": product.name,
            "price": str(product.price), "stock": product.stock,
            "category": product.category, "description": product.description
        }
    
    return await cached(cache_key, ttl=300, fetch_fn=fetch)


@app.put("/products/{product_id}")
async def update_product(product_id: int, data: dict, db: AsyncSession = Depends(get_db)):
    product = await db.get(Product, product_id)
    for key, value in data.items():
        setattr(product, key, value)
    await db.commit()
    
    await invalidate(f"product:{product_id}")
    await invalidate_pattern("products:limit:*")
    
    return {"status": "updated"}
'''

print(endpoints_example)

In [ ]:
# Проверка: Redis запущен в Docker?
result = subprocess.run(
    ["docker", "compose", "ps", "redis"],
    capture_output=True, text=True
)
print(result.stdout)

result = subprocess.run(
    ["docker", "compose", "exec", "-T", REDIS_CONTAINER, "redis-cli", "ping"],
    capture_output=True, text=True
)
if "PONG" in result.stdout:
    print("✓ Redis доступен")
else:
    print("✗ Redis недоступен:", result.stderr)

## 9. Практика: кеширование каталога товаров

### 9.1. Проверка работы кеша

Выполним запрос к `/products` несколько раз и убедимся, что данные берутся из кеша.

In [ ]:
def timed_request(url: str) -> tuple:
    """Возвращает (latency_ms, response)."""
    start = time.time()
    response = requests.get(url, timeout=10)
    elapsed = (time.time() - start) * 1000
    return round(elapsed, 2), response


# Очистим кеш для чистого эксперимента
subprocess.run(
    ["docker", "compose", "exec", "-T", REDIS_CONTAINER, "redis-cli", "FLUSHDB"],
    capture_output=True, text=True
)
print("✓ Кеш очищен")

# Холодный кеш: первый запрос
print("\n=== ХОЛОДНЫЙ КЕШ (первый запрос) ===")
latency_1, response_1 = timed_request(f"{NGINX_URL}/products?limit=50")
print(f"Latency: {latency_1} мс | Status: {response_1.status_code}")

# Тёплый кеш: последующие 5 запросов
print("\n=== ТЁПЛЫЙ КЕШ (5 запросов) ===")
warm_latencies = []
for i in range(5):
    latency, response = timed_request(f"{NGINX_URL}/products?limit=50")
    warm_latencies.append(latency)
    print(f"Запрос {i+1}: {latency} мс")

avg_warm = np.mean(warm_latencies)
print(f"\nСредняя latency тёплого кеша: {avg_warm:.2f} мс")
print(f"Ускорение: {latency_1 / avg_warm:.1f}×")

In [ ]:
# Визуализация холодного vs тёплого кеша
fig, ax = plt.subplots(figsize=(10, 5))

labels = ["Холодный\n(1-й запрос)"] + [f"Тёплый\n#{i+1}" for i in range(5)]
values = [latency_1] + warm_latencies
colors = ['#e74c3c'] + ['#2ecc71'] * 5

bars = ax.bar(labels, values, color=colors)
ax.set_ylabel('Latency (мс)')
ax.set_title('Холодный vs Тёплый кеш: latency запроса /products')
ax.axhline(y=avg_warm, color='gray', linestyle='--',
           label=f'Средняя тёплая: {avg_warm:.1f} мс')
ax.legend()

for bar, v in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
            f'{v:.1f}', ha='center', va='bottom', fontsize=11)

plt.tight_layout()
plt.show()

print(f"\nХолодный: {latency_1:.1f} мс")
print(f"Тёплый:   {avg_warm:.1f} мс")
print(f"Ускорение: {latency_1/avg_warm:.1f}×")

## 10. Практика: измерение Cache Hit Ratio

### 10.1. Метрики Redis

Redis собирает статистику по попаданиям и промахам. Запросим её:

In [ ]:
def redis_info(section: str = "stats") -> dict:
    """Возвращает секцию INFO из Redis как словарь."""
    result = subprocess.run(
        ["docker", "compose", "exec", "-T", REDIS_CONTAINER,
         "redis-cli", "INFO", section],
        capture_output=True, text=True
    )
    info = {}
    for line in result.stdout.split("\n"):
        if ":" in line and not line.startswith("#"):
            key, value = line.split(":", 1)
            info[key.strip()] = value.strip()
    return info


stats = redis_info("stats")
hits = int(stats.get("keyspace_hits", 0))
misses = int(stats.get("keyspace_misses", 0))
total = hits + misses
hit_ratio = (hits / total * 100) if total > 0 else 0

print("=" * 60)
print("CACHE HIT RATIO")
print("=" * 60)
print(f"Попаданий (hits):   {hits:>10,}")
print(f"Промахов (misses):  {misses:>10,}")
print(f"Всего запросов:     {total:>10,}")
print(f"Hit Ratio:          {hit_ratio:>9.2f}%")
print()

if hit_ratio > 95:
    print("✓ Отличный hit ratio — кеш работает эффективно")
elif hit_ratio > 80:
    print("✓ Хороший hit ratio")
elif hit_ratio > 50:
    print("⚠️  Средний hit ratio — увеличьте TTL или проверьте ключи")
else:
    print("✗ Низкий hit ratio — кеш неэффективен")

In [ ]:
# Информация о памяти и ключах
memory = redis_info("memory")
keyspace = redis_info("keyspace")

print("=" * 60)
print("ИНФОРМАЦИЯ О REDIS")
print("=" * 60)
print(f"Использовано памяти:  {memory.get('used_memory_human', 'N/A')}")
print(f"Пик памяти:           {memory.get('used_memory_peak_human', 'N/A')}")
print(f"Max memory:           {memory.get('maxmemory_human', 'N/A')}")
print(f"Max memory policy:    {memory.get('maxmemory_policy', 'N/A')}")
print()
print("Ключи по базам:")
for db_name, db_info in keyspace.items():
    print(f"  {db_name}: {db_info}")

# Все ключи в БД
result = subprocess.run(
    ["docker", "compose", "exec", "-T", REDIS_CONTAINER,
     "redis-cli", "KEYS", "*"],
    capture_output=True, text=True
)
keys = [k for k in result.stdout.strip().split("\n") if k]
print(f"\nВсего ключей: {len(keys)}")
print("Примеры ключей:")
for key in keys[:10]:
    print(f"  {key}")

## 11. Практика: нагрузочное тестирование с холодным и тёплым кешем

### 11.1. Эксперимент

1. Очистить кеш (`FLUSHDB`)
2. Запустить Locust с холодным кешем (60 секунд)
3. Прогреть кеш (несколько запросов к популярным эндпоинтам)
4. Запустить Locust с тёплым кешем (60 секунд)
5. Сравнить метрики

### 11.2. Холодный кеш

Выполните в терминале:

```bash
# Очистка кеша
docker compose exec redis redis-cli FLUSHDB

# Тест с холодным кешем
locust --host=http://localhost:8080 --csv=locust_cold --headless -u 100 -r 10 -t 60s
```

### 11.3. Тёплый кеш

```bash
# Прогрев кеша — делаем 20 запросов к популярным эндпоинтам
for i in $(seq 1 20); do
    curl -s http://localhost:8080/products?limit=50 > /dev/null
done

# Тест с тёплым кешем
locust --host=http://localhost:8080 --csv=locust_warm --headless -u 100 -r 10 -t 60s
```

In [ ]:
import glob

cold_file = glob.glob("locust_cold_stats.csv")
warm_file = glob.glob("locust_warm_stats.csv")

if cold_file and warm_file:
    df_cold = pd.read_csv(cold_file[0])
    df_warm = pd.read_csv(warm_file[0])
    print("✓ Загружены оба файла")
elif cold_file:
    df_cold = pd.read_csv(cold_file[0])
    df_warm = None
    print("⚠️  Загружен только cold. Запустите warm-тест.")
else:
    print("CSV-файлы не найдены. Создаём демонстрационные данные.")
    df_cold = pd.DataFrame({
        "Type": ["GET", "GET", "GET", ""],
        "Name": ["/health", "/products", "/products/1", "Aggregated"],
        "Request Count": [1500, 4200, 3800, 9500],
        "Failure Count": [0, 15, 8, 23],
        "50%": [3, 95, 45, 55],
        "95%": [12, 280, 150, 180],
        "99%": [35, 620, 380, 480],
        "Requests/s": [25.0, 70.0, 63.3, 158.3],
    })
    df_warm = pd.DataFrame({
        "Type": ["GET", "GET", "GET", ""],
        "Name": ["/health", "/products", "/products/1", "Aggregated"],
        "Request Count": [1500, 6800, 6200, 14500],
        "Failure Count": [0, 0, 0, 0],
        "50%": [3, 8, 6, 6],
        "95%": [10, 25, 18, 22],
        "99%": [30, 65, 45, 55],
        "Requests/s": [25.0, 113.3, 103.3, 241.7],
    })

print("\n=== ХОЛОДНЫЙ КЕШ ===")
display(df_cold)
print("\n=== ТЁПЛЫЙ КЕШ ===")
display(df_warm)

In [ ]:
# Сравнение метрик
df_cold_ep = df_cold[df_cold["Name"] != "Aggregated"].dropna(subset=["Name"])
df_warm_ep = df_warm[df_warm["Name"] != "Aggregated"].dropna(subset=["Name"])

merged = df_cold_ep.merge(
    df_warm_ep, on=["Type", "Name"], suffixes=("_cold", "_warm")
)

print("=" * 100)
print("СРАВНЕНИЕ: ХОЛОДНЫЙ vs ТЁПЛЫЙ КЕШ")
print("=" * 100)

print(f"{'Эндпоинт':<20} {'RPS cold':>10} {'RPS warm':>10} {'Δ RPS':>10} "
      f"{'p50 cold':>10} {'p50 warm':>10} {'Δ p50':>10}")
print("-" * 100)

for _, row in merged.iterrows():
    name = row["Name"]
    rps_cold = row["Requests/s_cold"]
    rps_warm = row["Requests/s_warm"]
    p50_cold = row["50%_cold"]
    p50_warm = row["50%_warm"]
    
    rps_delta = (rps_warm / rps_cold - 1) * 100 if rps_cold else 0
    p50_delta = (p50_warm / p50_cold - 1) * 100 if p50_cold else 0
    
    print(f"{name:<20} {rps_cold:>10.1f} {rps_warm:>10.1f} {rps_delta:>+9.0f}% "
          f"{p50_cold:>10.1f} {p50_warm:>10.1f} {p50_delta:>+9.0f}%")

In [ ]:
# Визуализация: cold vs warm
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

endpoints = merged["Name"].tolist()
x = np.arange(len(endpoints))
width = 0.35

# График 1: RPS
axes[0].bar(x - width/2, merged["Requests/s_cold"], width, label='Холодный', color='#e74c3c')
axes[0].bar(x + width/2, merged["Requests/s_warm"], width, label='Тёплый', color='#2ecc71')
axes[0].set_xlabel('Эндпоинт')
axes[0].set_ylabel('RPS')
axes[0].set_title('Пропускная способность (RPS)')
axes[0].set_xticks(x)
axes[0].set_xticklabels(endpoints, rotation=15)
axes[0].legend()

# График 2: p50
axes[1].bar(x - width/2, merged["50%_cold"], width, label='Холодный', color='#e74c3c')
axes[1].bar(x + width/2, merged["50%_warm"], width, label='Тёплый', color='#2ecc71')
axes[1].set_xlabel('Эндпоинт')
axes[1].set_ylabel('p50 (мс)')
axes[1].set_title('Медианная latency (p50)')
axes[1].set_xticks(x)
axes[1].set_xticklabels(endpoints, rotation=15)
axes[1].legend()

# График 3: p99
axes[2].bar(x - width/2, merged["99%_cold"], width, label='Холодный', color='#e74c3c')
axes[2].bar(x + width/2, merged["99%_warm"], width, label='Тёплый', color='#2ecc71')
axes[2].set_xlabel('Эндпоинт')
axes[2].set_ylabel('p99 (мс)')
axes[2].set_title('Хвостовая latency (p99)')
axes[2].set_xticks(x)
axes[2].set_xticklabels(endpoints, rotation=15)
axes[2].legend()

plt.tight_layout()
plt.show()

### 11.4. Что произошло «под капотом»

**Холодный кеш:**

1. Запрос приходит в FastAPI
2. `get_json(key)` возвращает `None` (промах)
3. Запрос идёт в PostgreSQL
4. PostgreSQL делает Seq Scan или Index Scan
5. Результат сериализуется в JSON
6. JSON записывается в Redis с TTL
7. Ответ отправляется клиенту

**Время:** 50–200 мс (зависит от сложности запроса)

**Тёплый кеш:**

1. Запрос приходит в FastAPI
2. `get_json(key)` возвращает данные из Redis
3. Ответ отправляется клиенту

**Время:** 1–5 мс (только сеть до Redis + десериализация)

**Ускорение:** в 20–50 раз по latency, в 3–5 раз по RPS.

## 12. Метрики Locust после кеширования

### 12.1. Что изменится в метриках

| Метрика | До кеша (модуль 3) | После кеша (модуль 4) | Причина |
|---|---|---|
| **RPS** | 100–150 | 300–500+ | Кеш снимает нагрузку с БД |
| **p50** | 20–50 мс | 3–10 мс | Чтение из памяти вместо диска |
| **p95** | 100–200 мс | 15–30 мс | Нет конкуренции за connection pool |
| **p99** | 200–1000 мс | 30–100 мс | Нет блокировок БД |
| **Failures** | Возможны | 0 | БД разгружена |

### 12.2. Анализ эффективности кеша

In [ ]:
# Итоговый анализ Cache Hit Ratio после нагрузочного теста
stats = redis_info("stats")
hits = int(stats.get("keyspace_hits", 0))
misses = int(stats.get("keyspace_misses", 0))
expired = int(stats.get("expired_keys", 0))
evicted = int(stats.get("evicted_keys", 0))

total = hits + misses
hit_ratio = (hits / total * 100) if total > 0 else 0

print("=" * 60)
print("ИТОГОВАЯ СТАТИСТИКА КЕША")
print("=" * 60)
print(f"Попаданий:          {hits:>10,}")
print(f"Промахов:           {misses:>10,}")
print(f"Истекло TTL:        {expired:>10,}")
print(f"Вытеснено (LRU):    {evicted:>10,}")
print(f"Hit Ratio:          {hit_ratio:>9.2f}%")
print()

if hit_ratio > 95:
    assessment = "Отлично — кеш работает очень эффективно"
elif hit_ratio > 80:
    assessment = "Хорошо — типичный hit ratio для production"
elif hit_ratio > 50:
    assessment = "Средне — проверьте TTL и паттерны доступа"
else:
    assessment = "Плохо — кеш не даёт эффекта"

print(f"Оценка: {assessment}")

if evicted > 0:
    print(f"\n⚠️  Внимание: {evicted} ключей вытеснено. Увеличьте maxmemory.")

### 12.3. Что делать, если кеш не помогает

| Симптом | Причина | Решение |
|---|---|---|
| Hit Ratio < 50% | Много уникальных ключей | Уменьшить разнообразие параметров, унифицировать запросы |
| `evicted_keys` растёт | Мало памяти под кеш | Увеличить `maxmemory` или использовать `allkeys-lfu` |
| Кеш есть, но latency высокая | Redis далеко или сеть медленная | Локальный Redis или connection pool |
| Кеш пустой после каждого запроса | TTL слишком маленький или ключи разные | Увеличить TTL, унифицировать ключи |
| Данные устаревают | Нет инвалидации | Добавить invalidate при update |
| Cache Stampede | TTL истекает, тысячи запросов идут в БД | Mutex, XFetch или stale-while-revalidate |

## 13. Выводы

### 13.1. Что мы узнали

1. **Кеширование — главный инструмент оптимизации чтения.** При 90% hit rate нагрузка на БД падает в 10 раз, а latency — в 5–20 раз.

2. **Уровни кеша комплементарны:** браузер → CDN → Nginx → Redis → БД. Каждый уровень отсекает часть трафика.

3. **Cache-Aside — базовый паттерн.** Простой, надёжный, покрывает 90% сценариев. Write-Through и Write-Behind — для специфических случаев.

4. **Инвалидация — самая сложная проблема.** TTL, explicit delete, version keys, event-driven — выбор зависит от требований к консистентности.

5. **Cache Stampede (dogpile)** — реальная угроза при истечении TTL. Решения: mutex, XFetch, stale-while-revalidate.

6. **Redis eviction policies** — `allkeys-lru` для универсального кеша, `allkeys-lfu` при неравномерной популярности ключей.

7. **Cache Hit Ratio** — главная метрика эффективности. > 80% — хорошо, > 95% — отлично. Проверяется через `INFO stats`.

8. **CDN** снижает latency для географически распределённых пользователей и разгружает origin-серверы.

### 13.2. Ограничения текущей конфигурации

| Ограничение | Как решить |
|---|---|
| Один Redis — SPOF | Redis Sentinel или Cluster |
| Нет репликации кеша | Redis Replication |
| Кеш локальный (не распределённый) | Redis Cluster |
| Нет метрик в Prometheus | redis_exporter |
| Нет CDN | Cloudflare, CloudFront |
| Инвалидация через `KEYS` | Использовать SCAN или versioned keys |

### 13.3. Что дальше

В **модуле 5** добавим Kafka для асинхронной обработки заказов. Кеш останется для чтения, а Kafka возьмёт на себя тяжёлые записи.

В **модуле 8** Redis станет частью Kubernetes-кластера, а Nginx Ingress Controller заменит standalone Nginx.

### 13.4. Контрольные вопросы

1. Почему Cache-Aside — самый популярный паттерн кеширования?
2. Что такое Cache Stampede и как его предотвратить?
3. Какая разница между `allkeys-lru` и `volatile-lru`?
4. Почему кеш с TTL 60 сек может отдавать устаревшие данные?
5. Как измерить эффективность кеша?
6. Почему Redis часто выбирают вместо Memcached?
7. Какие данные нельзя кешировать в CDN?
8. Что произойдёт, если Redis упадёт? Как система должна себя вести?
9. Почему `stale-while-revalidate` лучше, чем синхронное обновление?
10. Как связаны Cache Hit Ratio и latency?

In [ ]:
# Итоговая сводка
print("=" * 70)
print("ИТОГОВАЯ СВОДКА ПО МОДУЛЮ 4")
print("=" * 70)
print()
print("Архитектура:")
print("  Клиент → Nginx :8080 → 3 × FastAPI → Redis → PostgreSQL")
print()
print("Кеш-паттерн: Cache-Aside с mutex-защитой от stampede")
print("Ключи: product:{id}, products:limit:{N}")
print("TTL: 60 сек (каталог), 300 сек (карточка товара)")
print("Eviction: allkeys-lru, maxmemory 512mb")
print()
print("Проверено:")
print("  ✓ Redis добавлен в docker-compose")
print("  ✓ Cache-Aside реализован в эндпоинтах")
print("  ✓ Инвалидация при обновлении работает")
print("  ✓ Холодный vs тёплый кеш измерен")
print("  ✓ Cache Hit Ratio рассчитан")
print("  ✓ Нагрузочное тестирование проведено")
print()
print("Ключевые метрики:")
print(f"  Cache Hit Ratio: {hit_ratio:.1f}% (норма > 80%)")
print(f"  Ускорение latency: в 5-20 раз")
print(f"  Рост RPS: в 2-5 раз")
print(f"  Evicted keys: {evicted} (норма 0)")
print()
print("Следующий шаг: модуль 5 — асинхронность, очереди и EDA")
print("=" * 70)